# Week 3-2 ? SFM-03 Resource Addendum Practice

This addendum keeps the original practice notebook unchanged and adds the workbook-only details found during validation: the official Bessel sample, the transcript-versus-slide risk limit comparison, and the Stooq practice exercise answer-key workflow. Every dataset is loaded by a relative path from this folder.

## 1. Setup and editable configuration

Change the file names or start price here if you want to reuse this notebook with another local dataset. The Monte Carlo section uses a fixed random seed so the visible outputs are reproducible.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

# EDIT HERE
DATA_DIR = Path(".")
PORTFOLIO_FILE = DATA_DIR / "portfolio_sbi_icici.csv"
NIFTY_FILE = DATA_DIR / "nifty.csv"
PRACTICE_RETURNS_FILE = DATA_DIR / "sfm03_practice_nifty_returns.csv"
PRACTICE_ANSWER_FILE = DATA_DIR / "sfm03_practice_answer_key.csv"

# Lecture transcript mentions a tight investor limit around 1.03%; the slide deck states 2.7%.
RISK_LIMITS = {"transcript_1_03pct": 0.0103, "transcript_1_303pct": 0.01303, "slides_2_7pct": 0.027}
MONTE_CARLO_START_PRICE = 18065.00  # Lecture NIFTY last price; replace with the exercise last NIFTY price if you have it.
N_SIM = 5000
DAYS = 10
SEED = 20260302

print("Configured files:")
for p in [PORTFOLIO_FILE, NIFTY_FILE, PRACTICE_RETURNS_FILE, PRACTICE_ANSWER_FILE]:
    print(" -", p)
print("Risk limits:", RISK_LIMITS)
print("Monte Carlo:", {"start_price": MONTE_CARLO_START_PRICE, "simulations": N_SIM, "days": DAYS, "seed": SEED})

Configured files:
 - portfolio_sbi_icici.csv
 - nifty.csv
 - sfm03_practice_nifty_returns.csv
 - sfm03_practice_answer_key.csv
Risk limits: {'transcript_1_03pct': 0.0103, 'transcript_1_303pct': 0.01303, 'slides_2_7pct': 0.027}
Monte Carlo: {'start_price': 18065.0, 'simulations': 5000, 'days': 10, 'seed': 20260302}


## 2. Load lecture data and official practice extracts

The original lecture data remains the SBIN/ICICI Bank portfolio file and ten-year NIFTY file. The addendum also loads the official practice worked-solution returns and one-line answer key extracted from the Excel workbook.

In [2]:
portfolio = pd.read_csv(PORTFOLIO_FILE, parse_dates=["Date"])
nifty = pd.read_csv(NIFTY_FILE, parse_dates=["Date"])
practice_returns = pd.read_csv(PRACTICE_RETURNS_FILE, parse_dates=["Date"])
answer_key = pd.read_csv(PRACTICE_ANSWER_FILE)

print("Lecture portfolio data:", portfolio.shape)
display(portfolio.head())
print("Lecture NIFTY data:", nifty.shape)
display(nifty.tail())
print("Official practice return extract:", practice_returns.shape)
display(practice_returns.head())
print("Official practice answer key:")
display(answer_key)

Lecture portfolio data: (60, 3)


,Date,SBIN,ICICIBANK
0,2023-01-30,538.200012,823.500000
1,2023-01-31,553.500000,831.900024
2,2023-02-01,527.349976,847.950012
3,2023-02-02,528.099976,857.900024
4,2023-02-03,544.200012,863.799988


Lecture NIFTY data: (2455, 2)


,Date,NIFTY
2450,2023-04-24,17743.400391
2451,2023-04-25,17769.250000
2452,2023-04-26,17813.599609
2453,2023-04-27,17915.050781
2454,2023-04-28,18065.000000


Official practice return extract: (158, 2)


,Date,Daily_Returns_NIFTY
0,2025-07-08,NaN
1,2025-07-09,-0.000835
2,2025-07-10,-0.003843
3,2025-07-11,-0.006206
4,2025-07-14,-0.000675


Official practice answer key:


,Question_ID,Question,Official_Worked_Answer
0,Q2,Identify the minimum price of SBIN using MIN().,62.27
1,Q3,Return the date corresponding to the minimum p...,2025-07-09 00:00:00
2,Q4,Compute the mean price of SBIN using AVERAGE().,69.7967088607595
3,Q5,Compute the mean price of HDB using AVERAGE().,35.61459556962026
4,Q6,Compute the sample standard deviation of SBIN ...,4.810624447070153
5,Q7,Compute the sample standard deviation of HDB p...,1.702721402547587
6,Q8,Compute the standard deviation of daily return...,0.007667194761460805
7,Q9,Compute the average of daily returns of NIFTY ...,-0.0001291406931850168
8,Q10,Calculate the variance of daily returns using ...,5.8785875510172e-05


## 3. Official Bessel-correction sample

The official explanation workbook uses the sample `[2051, 2053, 2055, 2050, 2051]`, a known population mean of `2050`, and a sample mean of `2052`. Dividing sample squared deviations by `n` gives `3.2`; after Bessel's correction, dividing by `n - 1` gives `4.0`.

In [3]:
official_sample = np.array([2051, 2053, 2055, 2050, 2051], dtype=float)
population_mean = 2050.0
sample_mean = official_sample.mean()

bessel_table = pd.DataFrame({
    "Xi": official_sample,
    "Xi - population_mean": official_sample - population_mean,
    "(Xi - population_mean)^2": (official_sample - population_mean) ** 2,
    "Xi - sample_mean": official_sample - sample_mean,
    "(Xi - sample_mean)^2": (official_sample - sample_mean) ** 2,
})
population_variance = bessel_table["(Xi - population_mean)^2"].sum() / len(official_sample)
uncorrected_sample_variance = bessel_table["(Xi - sample_mean)^2"].sum() / len(official_sample)
corrected_sample_variance = bessel_table["(Xi - sample_mean)^2"].sum() / (len(official_sample) - 1)

summary = pd.DataFrame({
    "Quantity": ["population mean", "sample mean", "population variance", "sample variance divided by n", "sample variance with n-1"],
    "Value": [population_mean, sample_mean, population_variance, uncorrected_sample_variance, corrected_sample_variance],
})

display(bessel_table)
display(summary)
print(f"Official workbook check: population variance={population_variance:.1f}, uncorrected sample variance={uncorrected_sample_variance:.1f}, corrected sample variance={corrected_sample_variance:.1f}")

,Xi,Xi - population_mean,(Xi - population_mean)^2,Xi - sample_mean,(Xi - sample_mean)^2
0,2051.0,1.0,1.0,-1.0,1.0
1,2053.0,3.0,9.0,1.0,1.0
2,2055.0,5.0,25.0,3.0,9.0
3,2050.0,0.0,0.0,-2.0,4.0
4,2051.0,1.0,1.0,-1.0,1.0


,Quantity,Value
0,population mean,2050.0
1,sample mean,2052.0
2,population variance,7.2
3,sample variance divided by n,3.2
4,sample variance with n-1,4.0


Official workbook check: population variance=7.2, uncorrected sample variance=3.2, corrected sample variance=4.0


## 4. Chart the Bessel correction

The visual point is that changing the mean from the known population mean to the estimated sample mean changes the deviations, and then `n - 1` corrects the downward bias caused by estimating the mean from the same data.

In [4]:
fig, ax = plt.subplots(figsize=(7.2, 4.0))
labels = [f"x{i+1}" for i in range(len(official_sample))]
x = np.arange(len(labels))
width = 0.36
ax.bar(x - width/2, bessel_table["(Xi - population_mean)^2"], width, label="vs population mean 2050", color="#2563eb")
ax.bar(x + width/2, bessel_table["(Xi - sample_mean)^2"], width, label="vs sample mean 2052", color="#f59e0b")
ax.axhline(corrected_sample_variance, color="#dc2626", linestyle="--", label="corrected sample variance = 4.0")
ax.set_title("Official Bessel Correction Example")
ax.set_ylabel("Squared deviation")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(loc="upper left")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
out = DATA_DIR / "chart_addendum_2_sfm03_bessel_sample.png"
fig.savefig(out, dpi=160)
plt.show()
print("Saved", out)

Saved chart_addendum_2_sfm03_bessel_sample.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_23772\4148519553.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Portfolio risk limits: transcript versus slide deck

The same frontier can be judged against different volatility limits. The lecture transcript describes a tight risk-aversion example around `1.03%` daily volatility, while the official slide problem states `2.7%`. Under `2.7%`, every sampled SBIN/ICICI portfolio is feasible; under `1.03%`, only the low-risk side is feasible.

In [5]:
prices = portfolio.set_index("Date")[["SBIN", "ICICIBANK"]]
returns = np.log(prices).diff().dropna()
mu = returns.mean()
sigma = returns.std(ddof=1)
covariance = returns.cov().loc["SBIN", "ICICIBANK"]
correlation = returns.corr().loc["SBIN", "ICICIBANK"]

weights = np.linspace(0, 1, 101)
frontier = []
for w_sbin in weights:
    w_icici = 1 - w_sbin
    ret = w_sbin * mu["SBIN"] + w_icici * mu["ICICIBANK"]
    vol = np.sqrt(
        (w_sbin ** 2 * sigma["SBIN"] ** 2)
        + (w_icici ** 2 * sigma["ICICIBANK"] ** 2)
        + (2 * w_sbin * w_icici * covariance)
    )
    frontier.append({"Weight_SBIN": w_sbin, "Weight_ICICI": w_icici, "Daily_Vol": vol, "Daily_Return": ret})
frontier = pd.DataFrame(frontier)

limit_rows = []
for label, limit in RISK_LIMITS.items():
    feasible = frontier[frontier["Daily_Vol"] <= limit]
    if feasible.empty:
        best = None
        limit_rows.append({"Limit": label, "Vol_Limit": limit, "Feasible_Portfolios": 0, "Best_SBIN": np.nan, "Best_ICICI": np.nan, "Best_Return": np.nan, "Best_Vol": np.nan})
    else:
        best = feasible.loc[feasible["Daily_Return"].idxmax()]
        limit_rows.append({"Limit": label, "Vol_Limit": limit, "Feasible_Portfolios": len(feasible), "Best_SBIN": best.Weight_SBIN, "Best_ICICI": best.Weight_ICICI, "Best_Return": best.Daily_Return, "Best_Vol": best.Daily_Vol})
limit_summary = pd.DataFrame(limit_rows)
min_vol = frontier.loc[frontier["Daily_Vol"].idxmin()]

print("Mean daily log returns:")
display(mu.to_frame("mean"))
print("Sample daily volatility:")
display(sigma.to_frame("stdev"))
print(f"Covariance={covariance:.8f}; correlation={correlation:.4f}")
print("Minimum-volatility sampled portfolio:")
display(min_vol.to_frame().T)
print("Best feasible portfolio under each source's limit:")
display(limit_summary)

Mean daily log returns:


,mean
SBIN,0.001218
ICICIBANK,0.001835


Sample daily volatility:


,stdev
SBIN,0.016094
ICICIBANK,0.010450


Covariance=0.00006864; correlation=0.4081
Minimum-volatility sampled portfolio:


,Weight_SBIN,Weight_ICICI,Daily_Vol,Daily_Return
18,0.18,0.82,0.010103,0.001724


Best feasible portfolio under each source's limit:


,Limit,Vol_Limit,Feasible_Portfolios,Best_SBIN,Best_ICICI,Best_Return,Best_Vol
0,transcript_1_03pct,0.01030,26,0.05,0.95,0.001804,0.010282
1,transcript_1_303pct,0.01303,72,0.00,1.00,0.001835,0.010450
2,slides_2_7pct,0.02700,101,0.00,1.00,0.001835,0.010450


## 6. Chart the efficient-frontier source difference

The vertical lines show why the two limits should both be documented. A tighter limit changes the portfolio answer; a loose `2.7%` limit makes the all-ICICI portfolio feasible and therefore maximizes historical return among the two assets.

In [6]:
fig, ax = plt.subplots(figsize=(7.4, 4.2))
ax.plot(frontier["Daily_Vol"] * 100, frontier["Daily_Return"] * 100, color="#2563eb", lw=2.2, label="101 sampled portfolios")
ax.scatter(min_vol["Daily_Vol"] * 100, min_vol["Daily_Return"] * 100, s=80, color="#f59e0b", zorder=5, label="minimum volatility")
colors = {"transcript_1_03pct": "#dc2626", "transcript_1_303pct": "#7c3aed", "slides_2_7pct": "#16a34a"}
for _, row in limit_summary.iterrows():
    ax.axvline(row["Vol_Limit"] * 100, color=colors[row["Limit"]], linestyle="--", alpha=0.75, label=row["Limit"].replace("_", " "))
    if not np.isnan(row["Best_Vol"]):
        ax.scatter(row["Best_Vol"] * 100, row["Best_Return"] * 100, s=65, color=colors[row["Limit"]], edgecolor="black", linewidth=0.5)
ax.set_xlabel("Daily portfolio volatility (%)")
ax.set_ylabel("Mean daily log return (%)")
ax.set_title("SBIN/ICICI Frontier Under Transcript and Slide Risk Limits")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, loc="best")
fig.tight_layout()
out = DATA_DIR / "chart_addendum_1_sfm03_limit_comparison.png"
fig.savefig(out, dpi=160)
plt.show()
print("Saved", out)

Saved chart_addendum_1_sfm03_limit_comparison.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_23772\3558559115.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Official practice-exercise formula map

The practice exercise pack shifts from the lecture's SBIN/ICICI/NIFTY files to a Stooq workflow with `HDB`, `SBIN`, and `NIFTY`. The formulas are basic but important because they make the Excel answer cells auditable.

In [7]:
formula_map = pd.DataFrame([
    {"Task": "Daily NIFTY return", "Excel formula idea": "=(today - yesterday) / yesterday", "Python equivalent": "pct_change()"},
    {"Task": "Minimum SBIN price", "Excel formula idea": "=MIN(SBIN_range)", "Python equivalent": "sbin.min()"},
    {"Task": "Date of SBIN minimum", "Excel formula idea": "=XLOOKUP(min_price, SBIN_range, Date_range)", "Python equivalent": "idxmin() then Date lookup"},
    {"Task": "Mean price", "Excel formula idea": "=AVERAGE(price_range)", "Python equivalent": "mean()"},
    {"Task": "Sample standard deviation", "Excel formula idea": "=STDEV.S(range)", "Python equivalent": "std(ddof=1)"},
    {"Task": "Variance from stdev", "Excel formula idea": "=stdev_cell^2", "Python equivalent": "sigma**2"},
    {"Task": "10-day Monte Carlo", "Excel formula idea": "=previous_price*EXP(NORM.INV(RAND(), mean, std))", "Python equivalent": "price[t-1] * exp(normal(mean, std))"},
])
display(formula_map)
print("Official worked answer key values used for Q2-Q10:")
display(answer_key)

,Task,Excel formula idea,Python equivalent
0,Daily NIFTY return,=(today - yesterday) / yesterday,pct_change()
1,Minimum SBIN price,=MIN(SBIN_range),sbin.min()
2,Date of SBIN minimum,"=XLOOKUP(min_price, SBIN_range, Date_range)",idxmin() then Date lookup
3,Mean price,=AVERAGE(price_range),mean()
4,Sample standard deviation,=STDEV.S(range),std(ddof=1)
5,Variance from stdev,=stdev_cell^2,sigma**2
6,10-day Monte Carlo,"=previous_price*EXP(NORM.INV(RAND(), mean, std))","price[t-1] * exp(normal(mean, std))"


Official worked answer key values used for Q2-Q10:


,Question_ID,Question,Official_Worked_Answer
0,Q2,Identify the minimum price of SBIN using MIN().,62.27
1,Q3,Return the date corresponding to the minimum p...,2025-07-09 00:00:00
2,Q4,Compute the mean price of SBIN using AVERAGE().,69.7967088607595
3,Q5,Compute the mean price of HDB using AVERAGE().,35.61459556962026
4,Q6,Compute the sample standard deviation of SBIN ...,4.810624447070153
5,Q7,Compute the sample standard deviation of HDB p...,1.702721402547587
6,Q8,Compute the standard deviation of daily return...,0.007667194761460805
7,Q9,Compute the average of daily returns of NIFTY ...,-0.0001291406931850168
8,Q10,Calculate the variance of daily returns using ...,5.8785875510172e-05


## 8. Practice Monte Carlo with the official answer-key mean and volatility

The official practice workbook supplies the NIFTY daily-return mean and sample standard deviation. This section uses those answer-key values in the same `EXP(NORM.INV(RAND(), mean, std))` structure, with a fixed seed and an editable start price.

In [8]:
mean_return = float(answer_key.loc[answer_key["Question_ID"] == "Q9", "Official_Worked_Answer"].iloc[0])
stdev_return = float(answer_key.loc[answer_key["Question_ID"] == "Q8", "Official_Worked_Answer"].iloc[0])
variance_return = float(answer_key.loc[answer_key["Question_ID"] == "Q10", "Official_Worked_Answer"].iloc[0])
print(f"Practice answer-key mean={mean_return:.9f}, stdev={stdev_return:.9f}, variance={variance_return:.9e}")
print(f"Check stdev^2={stdev_return**2:.9e}")

rng = np.random.default_rng(SEED)
shocks = rng.normal(mean_return, stdev_return, size=(DAYS, N_SIM))
paths = np.empty((DAYS + 1, N_SIM))
paths[0] = MONTE_CARLO_START_PRICE
for day in range(1, DAYS + 1):
    paths[day] = paths[day - 1] * np.exp(shocks[day - 1])
final_prices = paths[-1]
mc_summary = pd.DataFrame({
    "Metric": ["start price", "mean final", "median final", "5th percentile VaR floor", "expected shortfall below 5th", "95th percentile"],
    "Value": [MONTE_CARLO_START_PRICE, final_prices.mean(), np.median(final_prices), np.percentile(final_prices, 5), final_prices[final_prices <= np.percentile(final_prices, 5)].mean(), np.percentile(final_prices, 95)],
})
display(mc_summary)

fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.6))
for i in range(80):
    axes[0].plot(range(DAYS + 1), paths[:, i], color="#2563eb", alpha=0.12, linewidth=0.8)
axes[0].plot(range(DAYS + 1), np.percentile(paths, 50, axis=1), color="#111827", linewidth=2.0, label="median path")
axes[0].set_title("10-Day Practice MC Paths")
axes[0].set_xlabel("Trading day")
axes[0].set_ylabel("Simulated price")
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)

var5 = np.percentile(final_prices, 5)
es5 = final_prices[final_prices <= var5].mean()
axes[1].hist(final_prices, bins=50, color="#0f766e", alpha=0.75)
axes[1].axvline(var5, color="#dc2626", linestyle="--", label=f"P5={var5:,.0f}")
axes[1].axvline(es5, color="#7c2d12", linestyle=":", label=f"ES={es5:,.0f}")
axes[1].set_title("Final Price Distribution")
axes[1].set_xlabel("Day-10 price")
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8)
fig.tight_layout()
out = DATA_DIR / "chart_addendum_3_sfm03_practice_mc.png"
fig.savefig(out, dpi=160)
plt.show()
print("Saved", out)

Practice answer-key mean=-0.000129141, stdev=0.007667195, variance=5.878587551e-05
Check stdev^2=5.878587551e-05


,Metric,Value
0,start price,18065.000000
1,mean final,18052.772745
2,median final,18050.969135
3,5th percentile VaR floor,17348.373140
4,expected shortfall below 5th,17164.628224
5,95th percentile,18783.016803


Saved chart_addendum_3_sfm03_practice_mc.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_23772\2210990872.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Bollinger formulas and breach count

The official notes define Bollinger Bands as a 20-period moving average plus or minus two 20-period sample standard deviations. This code confirms the formula on the lecture NIFTY data and counts how often the close falls outside the band.

In [9]:
bb = nifty.set_index("Date").copy()
bb["SMA20"] = bb["NIFTY"].rolling(20).mean()
bb["STD20"] = bb["NIFTY"].rolling(20).std(ddof=1)
bb["Upper"] = bb["SMA20"] + 2 * bb["STD20"]
bb["Lower"] = bb["SMA20"] - 2 * bb["STD20"]
bb["Outside"] = (bb["NIFTY"] > bb["Upper"]) | (bb["NIFTY"] < bb["Lower"])
valid = bb.dropna()
breach_rate = valid["Outside"].mean()
print(f"Bollinger valid rows={len(valid)}, outside-band observations={valid['Outside'].sum()}, outside-band rate={breach_rate:.2%}")
display(valid.tail()[["NIFTY", "SMA20", "STD20", "Upper", "Lower", "Outside"]])

Bollinger valid rows=2436, outside-band observations=223, outside-band rate=9.15%


,NIFTY,SMA20,STD20,Upper,Lower,Outside
Date,,,,,,
2023-04-24,17743.400391,17453.517676,307.081788,18067.681252,16839.354099,False
2023-04-25,17769.250000,17484.385156,306.194224,18096.773605,16871.996708,False
2023-04-26,17813.599609,17521.220117,298.817423,18118.854963,16923.585272,False
2023-04-27,17915.050781,17569.720117,278.400665,18126.521448,17012.918787,False
2023-04-28,18065.000000,17623.685156,263.439689,18150.564534,17096.805779,False


## 10. Addendum checklist

Covered here: official Bessel workbook numbers, source-difference risk limits, Stooq practice answer key, formula map, reproducible 10-day Monte Carlo, and exact Bollinger formulas. The original notebook remains preserved.